In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

def fz_loss(returns: np.ndarray, VaR: np.ndarray, ES: np.ndarray, quantile: float):
    """
    Calculates the FZ loss, as specified by Patton et al. (2019).
 
    Args:
        returns: Array of returns.
        VaR: Value at Risk estimates for the quantile level (negative number = loss).
        ES: Expected Shortfall estimates for the quantile level (negative number = loss).
        quantile: Quantile level (lower quantile => bigger loss). **NB**: Not confidence level.
    """
    L = (returns < VaR).astype(int)
    term1 = -L * (VaR - returns) / (quantile * ES)
    term2 = VaR / ES
    term3 = np.log(-ES)
    return term1 + term2 + term3 - 1

def al_loss(returns: np.ndarray, VaR: np.ndarray, ES: np.ndarray, quantile: float):
    """
    Calculates Assymetric Laplace Density log score, as introduced by Taylor (2017).
 
    Args:
        returns: Array of returns.
        VaR: Value at Risk estimates for the quantile level (negative number = loss).
        ES: Expected Shortfall estimates for the quantile level (negative number = loss).
        quantile: Quantile level. **NB**: Not confidence level.
    """
    L = (returns < VaR).astype(int)
    term1 = -np.log((quantile - 1) / ES)
    term2 = -(returns - VaR) * (quantile - L) / (quantile * ES)
    return term1 + term2


In [ ]:
VERSION = "IV_RV"         
WINDOW_SIZE = 2000     

PROJECT_ROOT = Path.cwd().parent
CSV_PATH = PROJECT_ROOT / "predictions" / f"LSTM_{VERSION}.csv"

DATE_COL = "Date"
TRUE_COL = "TrueY"

ALPHAS = [0.010, 0.025, 0.050, 0.950, 0.975, 0.990]

# ---------------- LOAD + CLEAN ----------------
if not CSV_PATH.exists():
    raise FileNotFoundError(f"File not found: {CSV_PATH.resolve()}")

df = pd.read_csv(CSV_PATH)

# Ensure required columns exist
required_cols = [DATE_COL, TRUE_COL]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column '{col}' in {CSV_PATH.name}")

# Convert types safely
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df[TRUE_COL] = pd.to_numeric(df[TRUE_COL], errors="coerce")

# Convert all Quantile_ and ES_ columns to numeric
for c in df.columns:
    if c.startswith("Quantile_") or c.startswith("ES_"):
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(DATE_COL)

print(f"\nLoaded file: {CSV_PATH.name}")
print(f"Rows: {len(df)}")
print(f"Date range: {df[DATE_COL].min()} → {df[DATE_COL].max()}")


Loaded file: LSTM_IV_RV_D.csv
Rows: 1711
Date range: 2016-02-05 00:00:00 → 2022-12-29 00:00:00


In [ ]:
# loss eval
returns = df[TRUE_COL]
results = []

for alpha_raw in ALPHAS:
    q_col = f"Quantile_{alpha_raw:.3f}"
    es_col = f"ES_{alpha_raw:.3f}"

    if q_col not in df.columns or es_col not in df.columns:
        print(f"Missing columns for alpha={alpha_raw}")
        continue

    VaR = df[q_col]
    ES  = df[es_col]
    r   = returns

    mask = np.isfinite(r) & np.isfinite(VaR) & np.isfinite(ES)
    r, VaR, ES = r[mask].to_numpy(), VaR[mask].to_numpy(), ES[mask].to_numpy()

    is_upper = alpha_raw > 0.5
    a_use = (1 - alpha_raw) if is_upper else alpha_raw

    if is_upper:
        r, VaR, ES = -r, -VaR, -ES

    valid_es = ES < 0
    r2, VaR2, ES2 = r[valid_es], VaR[valid_es], ES[valid_es]

    fz_mean = np.nanmean(fz_loss(r2, VaR2, ES2, quantile=a_use))
    al_mean = np.nanmean(al_loss(r2, VaR2, ES2, quantile=a_use))

    results.append({
        "alpha": alpha_raw,
        "tail": "right" if is_upper else "left",
        "n_total": int(mask.sum()),
        "n_used": int(valid_es.sum()),
        "FZ_mean": fz_mean,
        "AL_mean": al_mean
    })

results_df = pd.DataFrame(results)

print("\n=== FZ and AL Loss Summary ===")
print(results_df.round(6))


=== FZ and AL Loss Summary ===
   alpha   tail  n_total  n_used   FZ_mean   AL_mean
0  0.010   left     1711    1711 -4.298355 -3.291583
1  0.025   left     1711    1711 -4.492462 -3.471606
2  0.050   left     1711    1711 -4.651311 -3.604636
3  0.950  right     1711    1711 -4.658782 -3.603865
4  0.975  right     1711    1711 -4.527192 -3.498206
5  0.990  right     1711    1711 -4.360263 -3.347097
